In [ ]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import shutil
# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    #.config("spark.sql.catalog.lakehouse.io-impl",
     #       "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.io.ResolvingFileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")


In [ ]:
# Clear all checkpoints
shutil.rmtree("/tmp/checkpoints", ignore_errors=True)
print("Checkpoints cleared")

# Drop iceberg tables so they get recreated fresh
spark.sql("DROP TABLE IF EXISTS lakehouse.taxi.bronze")
spark.sql("DROP TABLE IF EXISTS lakehouse.taxi.silver")
spark.sql("DROP TABLE IF EXISTS lakehouse.taxi.gold")
print("Tables dropped")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze (
    kafka_key STRING,
    raw_value STRING,
    topic STRING,
    partition INT,
    offset BIGINT,
    kafka_timestamp TIMESTAMP
) USING iceberg
""")

In [ ]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [ ]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.show(5)

In [ ]:
# SEMINAR TASK 
# Consume a few messages from the topic using kafka-console-consumer.sh to verify they are there

# docker exec kafka sh -c "/opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic taxi-trips --from-beginning  --max-messages 5"


In [ ]:
# starts the query
bronze = (
    raw_stream
    .select(
        F.col("key").cast("string").alias("kafka_key"),
        F.col("value").cast("string").alias("raw_value"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp").alias("kafka_timestamp")
    )
)

query = (
    bronze.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/checkpoints/taxi_bronze")
    .toTable("lakehouse.taxi.bronze")
)

query.awaitTermination()

In [ ]:
# RUN THIS TO STOP THE QUERY
query.stop()

In [ ]:
spark.sql("SELECT count(*) FROM lakehouse.taxi.bronze").show()
spark.sql("SELECT * FROM lakehouse.taxi.bronze LIMIT 10").show()

In [ ]:
spark.conf.set(
    "spark.sql.catalog.lakehouse.io-impl",
    "org.apache.iceberg.io.ResolvingFileIO"
)

In [ ]:
# SILVER
spark.sql("""
    CREATE OR REPLACE TABLE lakehouse.taxi.silver (
        VendorID INT,
        tpep_pickup_datetime TIMESTAMP,
        tpep_dropoff_datetime TIMESTAMP,
        passenger_count DOUBLE,
        trip_distance DOUBLE,
        RatecodeID INT,
        store_and_fwd_flag BOOLEAN,
        PULocationID INT,
        DOLocationID INT,
        payment_type INT,
        fare_amount DOUBLE,
        extra DOUBLE,
        mta_tax DOUBLE,
        tip_amount DOUBLE,
        tolls_amount DOUBLE,
        improvement_surcharge DOUBLE,
        total_amount DOUBLE,
        congestion_surcharge DOUBLE,
        Airport_fee DOUBLE,
        cbd_congestion_fee DOUBLE,
        trip_duration_minutes INT,
        avg_speed_kmh DOUBLE,
        pickup_zone STRING,
        pickup_borough STRING,
        dropoff_zone STRING,
        dropoff_borough STRING,
        is_peak_hour BOOLEAN,
        kafka_timestamp TIMESTAMP
    ) USING iceberg
""")


trip_schema = """
    VendorID INT, tpep_pickup_datetime TIMESTAMP, tpep_dropoff_datetime TIMESTAMP,
    passenger_count DOUBLE, trip_distance DOUBLE, RatecodeID DOUBLE,
    store_and_fwd_flag STRING, PULocationID INT, DOLocationID INT,
    payment_type LONG, fare_amount DOUBLE, extra DOUBLE, mta_tax DOUBLE,
    tip_amount DOUBLE, tolls_amount DOUBLE, improvement_surcharge DOUBLE,
    total_amount DOUBLE, congestion_surcharge DOUBLE, Airport_fee DOUBLE,
    cbd_congestion_fee DOUBLE
"""

zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones_cleaned = (zones
    .filter(F.col("LocationID").isNotNull() & (F.col("LocationID") > 0))
    .filter(F.col("Borough").isNotNull() & (F.col("Borough") != ""))
    .filter(F.col("Zone").isNotNull() & (F.col("Zone") != ""))
    .dropDuplicates(["LocationID"])
)


zones_bc = zones_cleaned  

def process_silver_batch(batch_df, batch_id):
    parsed = (
        batch_df
        .select(F.from_json("raw_value", trip_schema).alias("d"), "kafka_timestamp")
        .select("d.*", "kafka_timestamp")
        .withColumn("passenger_count", F.col("passenger_count").cast("int"))
        .withColumn("RatecodeID", F.col("RatecodeID").cast("int"))
        .withColumn("payment_type", F.col("payment_type").cast("int"))
        .withColumn("store_and_fwd_flag", F.col("store_and_fwd_flag") == "Y")
        # Cleaning
        .filter(F.col("passenger_count").isNotNull() & (F.col("passenger_count") > 0))
        .filter(F.col("trip_distance") >= 0)
        .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
        .filter(F.col("RatecodeID").isin([1, 2, 3, 4, 5, 6, 99]))
        .filter(F.col("payment_type").isin([0, 1, 2, 3, 4, 5, 6]))
        .withColumn("trip_duration_minutes",
            ((F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60).cast("int"))
        .filter((F.col("trip_duration_minutes") > 0) & (F.col("trip_duration_minutes") < 1440))
        .withColumn("avg_speed_kmh",
            F.try_divide(F.col("trip_distance") * 1.60934, F.col("trip_duration_minutes") / 60))
        .filter((F.col("avg_speed_kmh") >= 2) & (F.col("avg_speed_kmh") <= 130))
        .dropDuplicates(["VendorID", "tpep_pickup_datetime", "PULocationID"])
        # Enrich
        .join(F.broadcast(zones_bc).select(
            F.col("LocationID").alias("PULocationID"),
            F.col("Zone").alias("pickup_zone"),
            F.col("Borough").alias("pickup_borough")
        ), on="PULocationID", how="left")
        .join(F.broadcast(zones_bc).select(
            F.col("LocationID").alias("DOLocationID"),
            F.col("Zone").alias("dropoff_zone"),
            F.col("Borough").alias("dropoff_borough")
        ), on="DOLocationID", how="left")
        .withColumn("is_peak_hour",
            F.hour("tpep_pickup_datetime").isin([7, 8, 17, 18]))
    )

    parsed.createOrReplaceTempView("silver_batch")
    batch_df.sparkSession.sql("""
        MERGE INTO lakehouse.taxi.silver AS t
        USING silver_batch AS s
        ON  t.VendorID             = s.VendorID
        AND t.tpep_pickup_datetime = s.tpep_pickup_datetime
        AND t.PULocationID         = s.PULocationID
        WHEN NOT MATCHED THEN INSERT *
    """)
    
  #  parsed.writeTo("lakehouse.taxi.silver").append()

silver_stream = (
    spark.readStream
    .format("iceberg")
 #   .option("stream-from-timestamp", "0")  
    .load("lakehouse.taxi.bronze")
)

query_silver = (
    silver_stream.writeStream
    .foreachBatch(process_silver_batch)
    .option("checkpointLocation", "/tmp/checkpoints/taxi_silver") 
    .trigger(processingTime="10 seconds")
    .start()
)

query_silver.awaitTermination()


In [ ]:
query_silver.stop()

In [ ]:

bronze_cnt = spark.table("lakehouse.taxi.bronze").count()
silver_cnt = spark.table("lakehouse.taxi.silver").count()
print(f"Bronze: {bronze_cnt} rows")
print(f"Silver: {silver_cnt} rows ({bronze_cnt - silver_cnt} bad records filtered)")
spark.sql("SELECT * FROM lakehouse.taxi.silver LIMIT 5").show(truncate=False)

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS lakehouse.taxi.gold (
        day DATE,
        pickup_zone STRING,
        trip_count LONG,
        avg_distance DOUBLE,
        avg_fare DOUBLE,
        avg_total DOUBLE,
        tip_rate_pct DOUBLE,
        total_revenue DOUBLE
    ) USING iceberg
    PARTITIONED BY (day)
""")

def upsert_gold(batch_df, batch_id):
    local_spark = batch_df.sparkSession
    gold_batch = (
        batch_df
        .withColumn("day", F.to_date("tpep_pickup_datetime"))
        .groupBy("day", "pickup_zone")
        .agg(
            F.count("*").alias("trip_count"),
            F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
            F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
            F.round(F.avg("total_amount"), 2).alias("avg_total"),
            F.round(
                F.sum(F.when(F.col("tip_amount") > 0, 1).otherwise(0)).cast("double")
                / F.count("*") * 100, 2
            ).alias("tip_rate_pct"),
            F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        )
    )
    gold_batch.createOrReplaceTempView("gold_batch")
    local_spark.sql("""
        MERGE INTO lakehouse.taxi.gold AS target
        USING gold_batch AS source
        ON target.day = source.day AND target.pickup_zone = source.pickup_zone
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

gold_source = (
    spark.readStream
    .format("iceberg")
    .load("lakehouse.taxi.silver")
)

query_gold = (
    gold_source.writeStream
    .foreachBatch(upsert_gold)
    .option("checkpointLocation", "/tmp/checkpoints/taxi_gold")
    .trigger(processingTime="30 seconds")
    .start()
)

query_gold.awaitTermination()

In [ ]:
query_gold.stop()

In [ ]:
gold_cnt = spark.table("lakehouse.taxi.gold").count()
print(f"Gold: {gold_cnt} rows")
spark.sql("SELECT * FROM lakehouse.taxi.gold LIMIT 5").show(truncate=False)

In [ ]:
before_bronze = spark.sql("SELECT count(*) as cnt FROM lakehouse.taxi.bronze").collect()[0]["cnt"]
before_silver = spark.sql("SELECT count(*) as cnt FROM lakehouse.taxi.silver").collect()[0]["cnt"]
before_gold   = spark.sql("SELECT count(*) as cnt FROM lakehouse.taxi.gold").collect()[0]["cnt"]

print(f"BEFORE RESTART")
print(f"Bronze: {before_bronze}")
print(f"Silver: {before_silver}")
print(f"Gold:   {before_gold}")

In [ ]:
after_bronze = spark.sql("SELECT count(*) as cnt FROM lakehouse.taxi.bronze").collect()[0]["cnt"]
after_silver = spark.sql("SELECT count(*) as cnt FROM lakehouse.taxi.silver").collect()[0]["cnt"]
after_gold   = spark.sql("SELECT count(*) as cnt FROM lakehouse.taxi.gold").collect()[0]["cnt"]

print(f"FTER RESTART")
print(f"Bronze: {after_bronze}")
print(f"Silver: {after_silver}")
print(f"Gold:   {after_gold}")

print(f"\nDUPLICATE CHECK")
print(f"Bronze delta: {after_bronze - before_bronze} (expected 0)")
print(f"Silver delta: {after_silver - before_silver} (expected 0)")
print(f"Gold delta:   {after_gold - before_gold}   (expected 0)")



In [ ]:
# VARIANT A: streaming 

# spark.sql("""
# CREATE TABLE IF NOT EXISTS lakehouse.taxi.gold (
#     day DATE,
#     pickup_zone STRING,
#     trip_count LONG,
#     avg_distance DOUBLE,
#     avg_fare DOUBLE,
#     avg_total DOUBLE,
#     tip_rate_pct DOUBLE,
#     total_revenue DOUBLE
#     ) USING iceberg
# PARTITIONED BY (day)
# """)

# gold_source = (
#     spark.readStream
#     .format("iceberg")
#     .option("stream-from-timestamp", "0")
#     .load("lakehouse.taxi.silver")
# )

# gold = (
#     gold_source
#     .withColumn("day", F.to_date("tpep_pickup_datetime"))
#     .groupBy("day", "pickup_zone")
#     .agg(
#         F.count("*").alias("trip_count"),
#         F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
#         F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
#         F.round(F.sum(F.when(F.col("tip_amount") > 0, 1).otherwise(0))
#                 / F.count("*") * 100, 2).alias("tip_rate_pct"),
#         F.round(F.avg("total_amount"), 2).alias("avg_total"),
#         F.round(F.sum("total_amount"), 2).alias("total_revenue"),
#     )
# )

#def upsert_gold(batch_df, batch_id):
#    batch_df.createOrReplaceTempView("gold_batch")
#    spark.sql("""
#        MERGE INTO lakehouse.taxi.gold AS target
#        USING gold_batch AS source
#        ON target.day = source.day AND target.pickup_zone = source.pickup_zone
#        WHEN MATCHED THEN UPDATE SET *
#        WHEN NOT MATCHED THEN INSERT *
#    """)
    
#query_gold = (
#    gold_stream.writeStream
#    .outputMode("update")     
#    .trigger(processingTime="1 minute")
#    .option("checkpointLocation", "s3://checkpoints/gold") see vaja ülev vaadata!! 
#    .foreachBatch(upsert_gold)
#    .start()
#)

# query_gold.awaitTermination()
